In [57]:
import pandas as pd
import numpy as np
import random

## Ejercicio 3

In [58]:
df = pd.read_csv('datos_mochila.csv')

valores = df['Valor'].values
pesos = df['Peso_kg'].values
long_cromosoma = len(df)
W_max = 400

TAMANO_POBLACION = 50 # tamaño de los P(t)
GENERACIONES = 100 # criterio de parada
P_CRUZA = 0.8 # intensificacion, 80%: crean hijos y el 20% se quedan
P_MUTACION = 0.05 # diversificacion

# cromosoma binario aleatorio
def inicializacion():
    return [random.randint(0, 1) for _ in range(long_cromosoma)]

In [59]:
# calculo de fitness con penalizacion
def calcular_fitness(individuo):
    valor_total = np.sum(np.array(individuo) * valores)
    peso_total = np.sum(np.array(individuo) * pesos)
    
    if peso_total > W_max:
        # Penalización: Restamos 1000 puntos del fitness por cada kg excedente
        penalizacion = (peso_total - W_max) * 1000
        return max(0, valor_total - penalizacion)
    return valor_total

# selección: operador de Torneo (k=3)
def seleccion_torneo(poblacion, aptitudes, k=3):
    participantes = random.sample(list(zip(poblacion, aptitudes)), k)
    # Ordenar por aptitud descendente y retornar el cromosoma del ganador
    participantes.sort(key=lambda x: x[1], reverse=True)
    return participantes[0][0]

# cruza: operador 1-punto
def cruzar(padre1, padre2):
    if random.random() < P_CRUZA:
        punto_cruza = random.randint(1, long_cromosoma - 1)
        hijo1 = padre1[:punto_cruza] + padre2[punto_cruza:]
        hijo2 = padre2[:punto_cruza] + padre1[punto_cruza:]
        return hijo1, hijo2
    return padre1.copy(), padre2.copy()

# mutacion: operador Flip bit
def mutar(individuo):
    for i in range(long_cromosoma):
        if random.random() < P_MUTACION:
            individuo[i] = 1 - individuo[i] # invertimos bits 0 a 1 o 1 a 0
    return individuo

In [60]:
# P0
poblacion = [inicializacion() for _ in range(TAMANO_POBLACION)]

mejor_solucion_global = None
mejor_fitness_global = -1

for generacion in range(GENERACIONES):
    # Evaluar la población actual
    aptitudes = [calcular_fitness(ind) for ind in poblacion]
    
    # Identificar al mejor de esta generación
    mejor_idx = np.argmax(aptitudes)
    if aptitudes[mejor_idx] > mejor_fitness_global:
        mejor_fitness_global = aptitudes[mejor_idx]
        mejor_solucion_global = poblacion[mejor_idx].copy()
        
    nueva_poblacion = []
    
    # Elitismo: Conservar al mejor individuo en la nueva generación
    nueva_poblacion.append(mejor_solucion_global)
    
    # Generar el resto de los descendientes
    while len(nueva_poblacion) < TAMANO_POBLACION:
        # Selección
        padre1 = seleccion_torneo(poblacion, aptitudes)
        padre2 = seleccion_torneo(poblacion, aptitudes)
        
        # Reproducción
        hijo1, hijo2 = cruzar(padre1, padre2)
        
        # Variación
        hijo1 = mutar(hijo1)
        hijo2 = mutar(hijo2)
        
        nueva_poblacion.extend([hijo1, hijo2])
        
    # Ajustar tamaño si nos pasamos por 1 (debido al elitismo + pares de hijos)
    poblacion = nueva_poblacion[:TAMANO_POBLACION]

In [61]:
peso_final = np.sum(np.array(mejor_solucion_global) * pesos)

print("RESULTADOS DEL GA:")
print(f"Mejor vector binario - Cromosoma: {mejor_solucion_global}")
print(f"Fitness alcanzado: {mejor_fitness_global}")
print(f"Peso ocupado: {peso_final} kg / {W_max} kg\n")

df['Seleccionado GA'] = mejor_solucion_global
objetos_elegidos = df[df['Seleccionado GA'] == 1]
print("Artículos cargados en el contenedor:")
print(objetos_elegidos[['ID', 'Objeto', 'Valor', 'Peso_kg']].to_string(index=False))

RESULTADOS DEL GA:
Mejor vector binario - Cromosoma: [1, 1, 1, 1, 1, 0, 1, 0, 1, 1, 1, 0, 0, 1, 1, 0, 1, 1, 1, 1]
Fitness alcanzado: 2870
Peso ocupado: 398 kg / 400 kg

Artículos cargados en el contenedor:
 ID                   Objeto  Valor  Peso_kg
  1               Laptop Pro    150       15
  2      Generador Eléctrico    450       80
  3      Kit de Herramientas     60       12
  4              Panel Solar    200       40
  5            Batería Litio    180       35
  7      Drone de Inspección    220       18
  9             Impresora 3D    300       45
 10               Kit Médico    110       10
 11 Unidad de Almacenamiento     80       12
 14            Escáner Láser    310       30
 15         Módulo Satelital    280       42
 17           Cámara Térmica    160       14
 18      Microscopio Digital    140       18
 19        Router Industrial    100       15
 20          GPS Diferencial    130       12
